In [ ]:
import os
import gradio as gr
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import urllib.request

os.environ["OPENAI_API_KEY"]= ""

#urllib.request.urlretrieve('https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/2020_%EA%B2%BD%EC%A0%9C%EA%B8%88%EC%9C%B5%EC%9A%A9%EC%96%B4%20700%EC%84%A0_%EA%B2%8C%EC%8B%9C.pdf', filename='2020_경제금융용어 700선_게시.pdf')

#랭체인의 PDFLoader 를 이용해 PDF 파일을 로드함.
#이는 PDF 파일을 파이썬으로 읽고 싶을 때 가장 많이 쓰는 도구임
#PyPDFLoader(파일명)을 실행해 loader라는 객체를 선언하고 해당 객체를 통해 load_and_split()을 실행하면 PDF를 여러개의 청크로 분할한 문자열 리스트가 반환됨.
loader = PyPDFLoader("2020_경제금융용어 700선_게시.pdf")
texts = loader.load_and_split()
print('문서의 수 :', len(texts))

#texts[4]
# 결과값 형식을 보면 page_content 에는 분할된 텍스트의 본문이 저장돼 있고, source에는 해당 본문의 원문 파일의 이름이 저장되어 있음.
# 기본적으로 랭체인에서 PyPDFLoader() 객체를 선언하고, load_and_split()을 사용하는 경우에는 다음과 같은 형식을 따름.
# Document(page_content='내용', metadata={'source':파일명, 'page':페이지번호})

# 본문에 접근하고 싶다면 각 청크의(문자열 원소)에 .page_content 를 붙여서 호출하면 됨.
#print(texts[15].page_content)

# 이제 챗봇을 만들기 위해 챗봇에 필요없는 정보들을 삭제하기 위해 각 청크를 확인함.
#print(texts[0].page_content)

# 각 청크를 확인해 보면서 머리말 등과 같은 챗봇이 제공할 정보가 아니라고 판단되면 해당 청크를 삭제하는 것을 고려함.
# 0~12번 청크까지는 목차, 머리말 등이므로 삭제함.
# 이러한 판단은 원본 PDF파일과 PyPDFLoader() 객체를 선언하고, load_and_split()을 실행해서 얻은 청크들을 보고 챗봇의 개발자가 판단해야 함.
# 챗봇의 답변에 불필요한 정보들을 제거해야만 챗봇의 잘못된 답변을 막을 수있음.
# 다음 코드는 13번 청크부터 시작하도록 수정해서 texts 를 재저장한다는 의미임.
texts = texts[13:]
#print("줄어든 문서의 개수 : ", len(texts))

# 전처리가 정상적으로 수행되었음.
# 이번에는 뒤에 잇는 청크들을 확인하기 위해 마지막 청크를 출력해 보겠음. 파이썬 리스트에서 -1이 마지막 원소를 의미함.
# texts[-1]을 통해 마지막 청크의 내용을 확인해 보면 맺음말에 해당하므로 제거해야함.
# 마지막 청크를 제가한 후 정상적으로 제거됐는지 확인하기 위해서 청크의 수를 출력함.
texts = texts[:-1]
#print('마지막 데이터 제거한 후 문서의 수 : ', len(texts))

# 이제 금융 용어에 대한 설명으로만 구성된 352개의 청크를 모두 임베딩해서 벡터 데이터베이스로 적재해 보겠음.
# OpenAI의 Embedding API를 사용함. Chroma DB는 이 과정을 기능별로 이미 구현해 사용자가 벡터를 좀더 쉽게 다룰수 있도록 도와주는 편리한 벡터 응용도구임.
# Chroma.from_documents() 를 통해 벡터 데이터베이스 객체인 vectordb를 선언함.
# 이때 documents 에는 벡터화 단위가 될 텍스트 리스트를 매개변수로 사용하고,  에는 어떤 종류의 임베딩을 사용할 것인지 기재함.
embedding = OpenAIEmbeddings()

vectordb = Chroma.from_documents(
    documents = texts,
    embedding = embedding
)

# vectordb를 선언하고 나면, _collection 다음에 점을 찍고 다양한 함수들을 사용할 수잇음.
# count()는 현재 저장된 청크 또는 벡터 개수를 의미함.
print(vectordb._collection.count())

# 앞에서 확인했던 청크 개수인 352개와 동일함. 기본적으로 _collection.get()은 벡터 데이터베이스 객체인 vectordb 에 저장된 값들을 볼수 잇는 기능을 제공함.
# 어떤 값들을 호출 할 수 있는지 확인해 봄.
#for key in vectordb._collection.get():
#    print(key)

# vectordb 에 저장된 기존 청크들을 보고 싶다면 ['document']를 통해 불러올 수 잇음.
# vectordb 로부터 청크들을 다시 로드하고, 청크의 개수와 첫번째 청크를 출력해 보겠음.
documents = vectordb._collection.get()['documents']
#print('문서의 개수 :', len(documents))
print('---'*50)
#print('첫번째 문서 출력 :', documents[0])

# 임베딩 벡터의 값은 기본적으로 제공하지 않기 때문에 임베딩 벡터의 값도 확인하고 싶다면 get() 호출시 include=['embeddings']를 기재해야 함.
# 그 후 ['embeddings']를 통해 호출 할 수 잇음. 352개 청크의 임베딩 벡터를 불러와 embeddings 에 저장함.
# 그 후 임베딩 벡터의 개수를 출력해 352로 일치하는 지 확인함.

embeddings = vectordb._collection.get(include=['embeddings'])['embeddings']
#print('임베딩 벡터의 개수 :', len(embeddings))

# 첫번째 청크의 임베딩 벡터의 값과 첫번째 임베딩 벡터 값의 길이를 출력해 봄.
#print('첫번째 문서의 임베딩 값 출력: ', embeddings[0])
#print('첫번째 문서의 임베딩 값의 길이: ', len(embeddings[0]))

# OpenAI Embedding 을 사용하는 OpenAIEmbeddings()를 사용하면 각 청크는 1,536개의 숫자를 가진 벡터로 변환됨.
# 이번에는 metadatas 를 호출해 봄. metadatas 는 각 청크의 출처를 의미함.
metadatas = vectordb._collection.get()['metadatas']
#print('metadatas 의 개수:', len(metadatas))
#print('첫번째 문서의 출처:', metadatas[0])

# 벡터 도구들을 선언하고 나면 as_retriever()를 통해 입력된 텍스트로 부터 유사한 텍스트를 찾아주는 검색기 객체인 retriever를 선언 할 수있음.
# 그러고 나면 invoke(입력 텍스트)를 통해 입력된 텍스트와 유사한 청크들을 찾아서 반환.
# 이러한 과정을 통해 벡터의 유사도를 구하는 과정을 별도의 추가 구현 없이 손쉽게 사용할 수 있음.

# 검색기 객체인 retriever 를 선언할때 as_retriever()의 내부 값으로 search_kwargs={"k": 숫자}를 사용한다면 입력 텍스트로부터 유사한 청크를 검색할 때
# 유사도 순위 몇 위까지를 반환할 것인지 지정할 수 있음.
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

docs = retriever.invoke("비트코인이 궁금해")
print('유사 문서 갯수 :', len(docs))
print('---'* 20)
#print('첫번째 유사 문서 :', docs[0])
#print('두번째 유사 문서 :', docs[1])

# 검색 결과를 보면 첫번째 청크로 비트코인 용어에 대한 청크, 두번째 청크로 블록 체인 용어에 대한 청크가 검색되어 본문 중간에 비트코인이라는 단어가 언급됨.

# 이제 이러한 검색기를 ChatGPT와 연결하는 작업을 진행해 보겠음.
# 랭체인을 통해 ChatGPT를 연결할때, 프롬프트를 생성하는 방법으로 프롬프트 템플릿이 존재함.
# 프롬프트 템플릿을 이용해 ChatGPT에 사용할 프롬프트를 만들어 보겠음.

# 우선 해당 챗봇에게 한국 은행에서 만든 금융용어를 설명하는 챗봇으로 이름이 '금융쟁이'이며, '안상준'개발자가 제작했고, 주어진 검색 결과를 바탕으로만 답변하라는 지시문을 작성함.
# 이후 {context}는 앞으로 사용자의 입력 텍스트로 부터 검색된 결과가 들어갈 자리이며, {question}은 사용자의 입력 텍스트가 들어갈 자리임.
# 예를 들어 앞의 코드에서 '비트코인이  궁금해'는 {question}에 위치하게 되며, {context}의 위치에는 2개의 검색 결과가 들어가게 됨.
# 이렇게 만들어진 프롬프트 문자열을 PromptTemplate.from_template()의 입력으로 사용해 프롬프트 템플릿 객체인 prompt 를 만듬
template = """당신의 이름은 도른아이입니다. 한국은행에서 만든 금융 용어를 설명해주는 소규모 인공지능 입니다..
임명수 형님이 만들었습니다. 주어진 검색 결과를 바탕으로 답변하세요.
검색 결과가 없는 내용이라면 답변할 수 없다고 하세요. 한국의 경상도 지방 사투리를 사용하여 친근하게 답변하세요.
{context}

Question: {question}
Answer:
"""

prompt = PromptTemplate.from_template(template)

# 이제 ChatGPT 와 연결하기 위해 ChatOpenAI() 로 LLM 객체에 해당하는 llm을 만듬.
# 수많은 ChatGPT 모델 중 하나를 선택해 model_name 의 값으로 결정하면 되는데, 여기서는 GPT-4 버전 중에서 답변속도가 가장 빠른 gpt-4.1을 선택했음.
llm = ChatOpenAI(model_name="gpt-5.6", temperature=0)

# 이제 LLM 객체인 llm, 프롬프트 템플릿 객체인 prompt, 검색기 객체인 retrieve 를 연결함.
# 3 개의 객체를 연결하는 query_rag() 함수를 만듬.

# 이 함수는 질문을 입력 받아 RAG 파이프라인을 순차적으로 실행함.
# 먼저 retriever.invoke(question)으로 벡터 데이터베이스에서 질문과 의미적으로 유사한 문서들을 검색함.
# 검색된 docs 는 Document 객체들의 리스트이며, 각 Document 객체는  page_content속성에 실제 텍스트 내용을 담고 있음.

# 다음으로 "\n\n".join([doc.page_content for doc in docs]) 를 통해 검색된 모든 문서의 텍스트 내용을 추출하고, 각 문서 사이에서 빈 줄을 넣어 하나의 context 문자열로 병합함.
# 이렇게 만들어진 프롬프트를 llm.invoke(formatted_prompt)로 LLM에 전달해 응답을 생성함.

# 마지막으로 원본 질문(query), LLM 응답 텍스트(result), 그리고 검색에 사용된 원본 문서들(source_documents)을 딕셔너리 형태로 반환.
# source_documents 를 함께 반환함으로써 LLM이 어떤 문서를 근거로 답변했는지 추적할 수 있음.

def query_rag(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    formatted_prompt = prompt.format(question=question, context=context)
    response = llm.invoke(formatted_prompt)

    return {
        "query": question,
        "result": response.content,
        "source_documents": docs
    }

# 이제 query_rag를 통해 사용자의 입력으로 부터 금융용어와 관련된 챗봇의 답변을 얻을 수 있음.
# 임의의 '디커플링이란 무엇인가?'라는 텍스트를 입력해 query_rag 의 반환 결과를 확인해 보자.
input_text = "너는 뭘하는 챗봇이니?"
chatbot_response = query_rag(input_text)
#print(chatbot_response)

# 이번에는 Gradio 로 챗봇 UI를 구현해 보겠음.
# Gradio는 AI모델을 웹 형태로 배포할 수있게 돕는 파이썬 라이브러리임.
# 실제로 코드 몇 줄만으로 웹 기반 인터페이스를 구현할 수있음.

# Gradio 공식문서에서는 Gradio로 챗봇을 구현할 수 있도록 기본 코드를 제공함.
# https://www.gradio.app/guides/creating-a-chatbot

# 인터페이스 생성.
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="도른아이")
    msg = gr.Textbox(label="질문해 주세요")
    clear = gr.Button("대화 초기화")

    # 챗봇의 답변을 처리하는 함수.
    def respond(message, chat_history):
        result = query_rag(message)
        bot_message = result["result"]

        # 채팅 기록에 사용자의 메시지와 봇의 응답을 추가
        #chat_history.append((message, bot_message))
        # 핵심: 최신 포맷에 맞춰 dict 형태로 append , Gradio 최신 버전(5~6)에서는 role과 content를 가진 딕셔너리 형태의 리스트만 지원하도록 엄격해짐.
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_message})
        return "", chat_history

    # 사용자의 입력을 제출(submit)하면 respond 함수가 호출.
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

    # 초기화 버튼을 클릭하면 채팅 기록 간소화
    clear.click(lambda: None, None, chatbot, queue=False)

# 인터페이스 실행.
demo.launch(debug=True)

문서의 수 : 366
1408
------------------------------------------------------------------------------------------------------------------------------------------------------
유사 문서 갯수 : 2
------------------------------------------------------------
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
